# Same Estimate Different Uncertainty

*A companion Colab for the blog post.*

This notebook uses a toy 2x2 survival dataset to show the numerical relationship between:

1. a manually calculated **delta-delta mean**,
2. a **DABEST delta-delta mean difference** with a bootstrap confidence interval,
3. a linear regression **interaction beta**, and
4. a two-way ANOVA **interaction p value**.

The point is not that these methods are identical in every possible analysis.  
The point is narrower and more useful:

> In a clean 2x2 ordinary-least-squares setting, the delta-delta estimate and the regression interaction coefficient can be the same arithmetic contrast.  
> The difference is how uncertainty is generated.

DABEST is the main estimation tool in this notebook. Statsmodels is used only to compare the model-based interaction coefficient and ANOVA table.

**Model/mode note:** drafted with GPT-5.5 Thinking.


## 0. Install dependencies

Run this cell in Colab.

If Colab reports an import error after installing or upgrading dependencies, restart the runtime and continue from the next cell.


In [ ]:
%pip install -q dabest statsmodels seaborn

## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm

import dabest
import statsmodels.api as sm
from statsmodels.formula.api import ols

from IPython.display import display

print(f"DABEST version: {dabest.__version__}")

## 2. Simulate a 2x2 dataset

This is adapted from an older simulation I used for a delta-delta example.

There are two genotypes:

- `W`: wild type / control genotype
- `M`: mutant / experimental genotype

There are two treatments:

- `Placebo`
- `Drug`

The outcome here is `Survival_years`.

One small change from the original draft: the function now uses the `seed` argument. The original version accepted `seed`, but then hard-coded `np.random.seed(1000)`.


In [ ]:
def simulate_dataset(seed=1000, effects=[0, -4, -0.2, 0.5], widths=[1, 1, 0.9, 1]):
    """Simulate a 2x2 dataset for delta-delta demonstration.

    Cell order is:
    1. W, Placebo
    2. M, Placebo
    3. W, Drug
    4. M, Drug

    effects and widths are applied to those four blocks.
    """
    np.random.seed(seed)

    # Create samples.
    N1 = 50

    survival_drug1 = widths[0] * norm.rvs(loc=15, scale=4, size=N1 * 4) + effects[0]
    survival_drug1[N1:2*N1] = widths[1] * survival_drug1[N1:2*N1] + effects[1]
    survival_drug1[2*N1:3*N1] = widths[2] * survival_drug1[2*N1:3*N1] + effects[2]
    survival_drug1[3*N1:4*N1] = widths[3] * survival_drug1[3*N1:4*N1] + effects[3]

    # A second outcome, included because the original simulation had it.
    np.random.seed(seed + 8999)
    t_size_drug1 = norm.rvs(loc=3, scale=0.4, size=N1 * 4)
    t_size_drug1[N1:2*N1] = t_size_drug1[N1:2*N1] + 1.5
    t_size_drug1[2*N1:3*N1] = t_size_drug1[2*N1:3*N1] - 0.25

    treatment = (
        np.repeat("Placebo", N1 * 2).tolist()
        + np.repeat("Drug", N1 * 2).tolist()
    )

    genotype = (
        np.repeat("W", N1).tolist()
        + np.repeat("M", N1).tolist()
        + np.repeat("W", N1).tolist()
        + np.repeat("M", N1).tolist()
    )

    # Add an ID column. This makes paired plotting possible if needed,
    # but this notebook treats the main survival analysis as unpaired.
    id_values = list(range(0, N1 * 2))
    id_col = id_values + id_values

    df = pd.DataFrame({
        "ID": id_col,
        "Genotype": genotype,
        "Treatment": treatment,
        "Survival_years": survival_drug1,
        "Tumor_size_cm": t_size_drug1,
    })

    # Make category order explicit for plotting and regression.
    df["Genotype"] = pd.Categorical(df["Genotype"], categories=["W", "M"], ordered=True)
    df["Treatment"] = pd.Categorical(df["Treatment"], categories=["Placebo", "Drug"], ordered=True)
    df["GenoTreat"] = df["Genotype"].astype(str) + ", " + df["Treatment"].astype(str)

    return df


df_delta2_drug1 = simulate_dataset(
    seed=1000,
    effects=[0, -4, -0.2, 0.5],
    widths=[1, 1, 0.9, 1],
)

display(df_delta2_drug1.head())
display(df_delta2_drug1.groupby(["Genotype", "Treatment"], observed=True)["Survival_years"].agg(["mean", "std", "count"]))


## 3. The raw 2x2 pattern

First look at the raw data.

The plot below is not meant to be the final DABEST plot yet. It is just the ordinary grouped-data view many people start from.


In [ ]:
palette = {
    "W, Placebo": "#4C78A8",
    "M, Placebo": "#4C78A8",
    "W, Drug": "#4C78A8",
    "M, Drug": "#4C78A8",
}

palette0 = {
    "W, Placebo": "lightgray",
    "M, Placebo": "lightgray",
    "W, Drug": "lightgray",
    "M, Drug": "lightgray",
}

fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=df_delta2_drug1,
    x="GenoTreat",
    y="Survival_years",
    ax=ax,
    width=0.4,
    palette=palette0,
    boxprops={"edgecolor": "silver", "linewidth": 1},
)

sns.swarmplot(
    data=df_delta2_drug1,
    x="GenoTreat",
    y="Survival_years",
    ax=ax,
    size=3,
    palette=palette,
)

ax.spines[["top", "right"]].set_visible(False)
ax.set_xticklabels(["W\nPlacebo", "M\nPlacebo", "W\nDrug", "M\nDrug"])
ax.set_ylabel("Survival (years)")
ax.set_xlabel("")
ax.set_title("Raw 2x2 survival data")

plt.show()


## 4. Manual delta-delta calculation

The delta-delta is just arithmetic on the four cell means.

One equivalent way to write it is:

\[
(M_{\mathrm{Drug}} - W_{\mathrm{Drug}}) - (M_{\mathrm{Placebo}} - W_{\mathrm{Placebo}})
\]

That is the change in the genotype gap under drug relative to placebo.

Another equivalent way to write it is:

\[
(M_{\mathrm{Drug}} - M_{\mathrm{Placebo}}) - (W_{\mathrm{Drug}} - W_{\mathrm{Placebo}})
\]

That is the difference between the drug response in `M` and the drug response in `W`.


In [ ]:
def get_cell_means(df, y="Survival_years"):
    means = (
        df.groupby(["Genotype", "Treatment"], observed=True)[y]
          .mean()
          .unstack()
          .loc[["W", "M"], ["Placebo", "Drug"]]
    )
    return means


def manual_delta_delta(df, y="Survival_years"):
    means = get_cell_means(df, y=y)

    w_placebo = means.loc["W", "Placebo"]
    w_drug = means.loc["W", "Drug"]
    m_placebo = means.loc["M", "Placebo"]
    m_drug = means.loc["M", "Drug"]

    w_drug_effect = w_drug - w_placebo
    m_drug_effect = m_drug - m_placebo

    placebo_genotype_gap = m_placebo - w_placebo
    drug_genotype_gap = m_drug - w_drug

    delta_delta_by_drug_response = m_drug_effect - w_drug_effect
    delta_delta_by_gap_change = drug_genotype_gap - placebo_genotype_gap

    summary = pd.DataFrame({
        "quantity": [
            "W drug effect",
            "M drug effect",
            "delta-delta: M drug effect - W drug effect",
            "genotype gap under placebo",
            "genotype gap under drug",
            "delta-delta: drug genotype gap - placebo genotype gap",
        ],
        "value": [
            w_drug_effect,
            m_drug_effect,
            delta_delta_by_drug_response,
            placebo_genotype_gap,
            drug_genotype_gap,
            delta_delta_by_gap_change,
        ],
    })

    return means, summary


cell_means, manual_summary = manual_delta_delta(df_delta2_drug1)

print("Cell means:")
display(cell_means)

print("Manual delta-delta arithmetic:")
display(manual_summary)


## 5. DABEST delta-delta

Now use DABEST to estimate the same delta-delta with a bootstrap confidence interval.

Here, `Treatment` is the experiment column, and `Genotype` is the grouping variable inside each experiment.

DABEST will calculate the genotype contrast within placebo, the genotype contrast within drug, and then the delta-delta between those contrasts.


In [ ]:
survival_drug1_unpaired_delta2 = dabest.load(
    data=df_delta2_drug1,
    x=["Genotype", "Genotype"],
    y="Survival_years",
    delta2=True,
    experiment="Treatment",
    experiment_label=["Placebo", "Drug"],
    x1_level=["W", "M"],
    resamples=5000,
    random_seed=12345,
)

survival_drug1_unpaired_delta2


In [ ]:
# DABEST estimation plot.
fig = survival_drug1_unpaired_delta2.mean_diff.plot(
    swarmplot_kwargs={"alpha": 1},
    raw_marker_size=2,
    contrast_marker_size=5,
    custom_palette={"W": "#1f77b4", "M": "#1f77b4"},
    raw_label="Survival (years)",
    contrast_label=r"$\Delta$ Survival (years)",
    delta2_label=r"$\Delta\Delta$ Survival (years)",
)

plt.show()


### Extract the DABEST result table

DABEST versions can expose slightly different result-table columns.  
This cell prints the full delta-delta results table and then extracts the most useful columns if they are available.


In [ ]:
def get_dabest_delta_delta_results(dabest_obj):
    """Return the DABEST delta-delta results dataframe."""
    try:
        return dabest_obj.mean_diff.delta_delta.results.copy()
    except AttributeError as exc:
        raise AttributeError(
            "Could not find dabest_obj.mean_diff.delta_delta.results. "
            "Inspect `dir(dabest_obj.mean_diff)` and the DABEST version."
        ) from exc


dabest_delta_delta_results = get_dabest_delta_delta_results(survival_drug1_unpaired_delta2)

print("Full DABEST delta-delta results table:")
display(dabest_delta_delta_results.T)

cols_to_show = [
    c for c in [
        "control", "test", "difference", "ci",
        "bca_low", "bca_high",
        "pct_low", "pct_high",
        "pvalue_permutation"
    ]
    if c in dabest_delta_delta_results.columns
]

print("Compact DABEST summary:")
display(dabest_delta_delta_results[cols_to_show])


## 6. Linear regression interaction beta

Now fit an ordinary least-squares model.

The model is:

```text
Survival_years ~ Genotype * Treatment
```

With `W` and `Placebo` as the reference levels, the interaction beta estimates:

\[
(M_{\mathrm{Drug}} - W_{\mathrm{Drug}}) - (M_{\mathrm{Placebo}} - W_{\mathrm{Placebo}})
\]

That is the same delta-delta contrast we calculated manually.


In [ ]:
formula = 'Survival_years ~ C(Genotype, levels=["W", "M"]) * C(Treatment, levels=["Placebo", "Drug"])'

ols_model = ols(formula, data=df_delta2_drug1).fit()

print(ols_model.summary())


In [ ]:
def find_interaction_term(params_or_index):
    for name in params_or_index:
        if ":" in str(name):
            return name
    raise ValueError("No interaction term found.")


interaction_term = find_interaction_term(ols_model.params.index)
interaction_beta = ols_model.params[interaction_term]
interaction_ci_low, interaction_ci_high = ols_model.conf_int().loc[interaction_term]
interaction_p = ols_model.pvalues[interaction_term]

ols_interaction_summary = pd.DataFrame({
    "term": [interaction_term],
    "estimate": [interaction_beta],
    "ci_low_model_based": [interaction_ci_low],
    "ci_high_model_based": [interaction_ci_high],
    "p_value_model_based": [interaction_p],
})

display(ols_interaction_summary)


## 7. ANOVA interaction p value

A two-way ANOVA asks the same interaction question:

> Does the effect of treatment depend on genotype?

But this gives a model-based **p value**, not a bootstrap confidence interval around the delta-delta effect size.


In [ ]:
anova_type2 = sm.stats.anova_lm(ols_model, typ=2)

print("Type II ANOVA:")
display(anova_type2)

anova_interaction_row = find_interaction_term(anova_type2.index)
anova_interaction_p = anova_type2.loc[anova_interaction_row, "PR(>F)"]

print(f"ANOVA interaction p value: {anova_interaction_p:.6g}")


## 8. Put the three approaches side-by-side

This table is the heart of the notebook.

The manual delta-delta and OLS interaction beta should match numerically, up to floating-point rounding.  
The DABEST delta-delta should also estimate the same contrast.

But the intervals and p values come from different machinery.


In [ ]:
def extract_bootstrap_array(row):
    """Try to extract the DABEST delta-delta bootstrap distribution from a result row."""
    candidate_columns = [
        "bootstraps_delta_delta",
        "bootstraps",
        "delta_delta_bootstraps",
    ]

    for col in candidate_columns:
        if col in row.index:
            arr = row[col]
            if isinstance(arr, (list, tuple, np.ndarray, pd.Series)):
                return np.asarray(arr, dtype=float)

    return None


dabest_row = dabest_delta_delta_results.iloc[0]

manual_dd = (
    manual_summary
    .query('quantity == "delta-delta: drug genotype gap - placebo genotype gap"')
    ["value"]
    .iloc[0]
)

dabest_estimate = dabest_row["difference"] if "difference" in dabest_row.index else np.nan
dabest_ci_low = dabest_row["bca_low"] if "bca_low" in dabest_row.index else np.nan
dabest_ci_high = dabest_row["bca_high"] if "bca_high" in dabest_row.index else np.nan
dabest_perm_p = dabest_row["pvalue_permutation"] if "pvalue_permutation" in dabest_row.index else np.nan

comparison = pd.DataFrame([
    {
        "method": "Manual delta-delta",
        "estimate": manual_dd,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "p_value": np.nan,
        "uncertainty_source": "No uncertainty; direct arithmetic on observed means",
    },
    {
        "method": "DABEST delta-delta",
        "estimate": dabest_estimate,
        "ci_low": dabest_ci_low,
        "ci_high": dabest_ci_high,
        "p_value": dabest_perm_p,
        "uncertainty_source": "Bootstrap CI from resampled effect-size distribution",
    },
    {
        "method": "OLS interaction beta",
        "estimate": interaction_beta,
        "ci_low": interaction_ci_low,
        "ci_high": interaction_ci_high,
        "p_value": interaction_p,
        "uncertainty_source": "Model-based SE and t distribution",
    },
    {
        "method": "ANOVA interaction",
        "estimate": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "p_value": anova_interaction_p,
        "uncertainty_source": "Model-based F test under no-interaction null",
    },
])

display(comparison)


## 9. Visualize the DABEST bootstrap distribution

This plot makes the uncertainty distinction concrete.

The DABEST bootstrap distribution is created by repeatedly resampling the observed data and recomputing the delta-delta.  
The regression interval is overlaid as a model-based comparison.


In [ ]:
boot = extract_bootstrap_array(dabest_row)

if boot is None:
    print("Could not find the bootstrap distribution column in the DABEST results table.")
    print("Inspect the full DABEST table printed above.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))

    ax.hist(boot, bins=40, alpha=0.7)

    ax.axvline(dabest_estimate, linewidth=2, label="DABEST delta-delta estimate")
    ax.axvline(dabest_ci_low, linestyle="--", linewidth=1.5, label="DABEST BCa CI")
    ax.axvline(dabest_ci_high, linestyle="--", linewidth=1.5)

    ax.axvline(interaction_beta, linestyle=":", linewidth=2, label="OLS interaction beta")
    ax.axvline(interaction_ci_low, linestyle="-.", linewidth=1.5, label="OLS model-based CI")
    ax.axvline(interaction_ci_high, linestyle="-.", linewidth=1.5)

    ax.set_title("Bootstrap distribution of the delta-delta")
    ax.set_xlabel("Delta-delta survival estimate")
    ax.set_ylabel("Bootstrap count")
    ax.legend()

    plt.show()


## 10. A messier version

The clean example is useful because the arithmetic is easy to see.

But real biological data are usually less polite.

Here we keep the same 2x2 structure, but make the data more uneven by changing widths, adding a few extreme observations, and making the sample sizes unbalanced.


In [ ]:
df_messy = simulate_dataset(
    seed=2026,
    effects=[0, -4, -0.2, 0.5],
    widths=[1.0, 1.2, 0.8, 1.7],
)

# Make sample sizes unbalanced.
keep = (
    df_messy.query('Genotype == "W" and Treatment == "Placebo"').sample(n=45, random_state=1).index.tolist()
    + df_messy.query('Genotype == "M" and Treatment == "Placebo"').sample(n=30, random_state=2).index.tolist()
    + df_messy.query('Genotype == "W" and Treatment == "Drug"').sample(n=38, random_state=3).index.tolist()
    + df_messy.query('Genotype == "M" and Treatment == "Drug"').sample(n=22, random_state=4).index.tolist()
)

df_messy = df_messy.loc[keep].copy()

# Add a few high observations in the M Drug group.
mask = (df_messy["Genotype"].astype(str) == "M") & (df_messy["Treatment"].astype(str) == "Drug")
outlier_idx = df_messy.loc[mask].sample(n=2, random_state=5).index
df_messy.loc[outlier_idx, "Survival_years"] += [10, 7]

display(df_messy.groupby(["Genotype", "Treatment"], observed=True)["Survival_years"].agg(["mean", "std", "count"]))

fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=df_messy,
    x="GenoTreat",
    y="Survival_years",
    ax=ax,
    width=0.4,
    palette=palette0,
    boxprops={"edgecolor": "silver", "linewidth": 1},
)

sns.swarmplot(
    data=df_messy,
    x="GenoTreat",
    y="Survival_years",
    ax=ax,
    size=3,
    palette=palette,
)

ax.spines[["top", "right"]].set_visible(False)
ax.set_xticklabels(["W\nPlacebo", "M\nPlacebo", "W\nDrug", "M\nDrug"])
ax.set_ylabel("Survival (years)")
ax.set_xlabel("")
ax.set_title("Messier 2x2 survival data")

plt.show()


### Repeat the DABEST and model comparison on the messy data


In [ ]:
messy_delta2 = dabest.load(
    data=df_messy,
    x=["Genotype", "Genotype"],
    y="Survival_years",
    delta2=True,
    experiment="Treatment",
    experiment_label=["Placebo", "Drug"],
    x1_level=["W", "M"],
    resamples=5000,
    random_seed=12345,
)

fig = messy_delta2.mean_diff.plot(
    swarmplot_kwargs={"alpha": 1},
    raw_marker_size=2,
    contrast_marker_size=5,
    custom_palette={"W": "#1f77b4", "M": "#1f77b4"},
    raw_label="Survival (years)",
    contrast_label=r"$\Delta$ Survival (years)",
    delta2_label=r"$\Delta\Delta$ Survival (years)",
)

plt.show()

messy_dabest_results = get_dabest_delta_delta_results(messy_delta2)
display(messy_dabest_results.T)


In [ ]:
messy_means, messy_manual_summary = manual_delta_delta(df_messy)
messy_model = ols(formula, data=df_messy).fit()
messy_interaction_term = find_interaction_term(messy_model.params.index)

messy_anova_type1 = sm.stats.anova_lm(messy_model, typ=1)
messy_anova_type2 = sm.stats.anova_lm(messy_model, typ=2)
messy_anova_type3 = sm.stats.anova_lm(messy_model, typ=3)

messy_row = messy_dabest_results.iloc[0]
messy_reg_ci_low, messy_reg_ci_high = messy_model.conf_int().loc[messy_interaction_term]

messy_comparison = pd.DataFrame([
    {
        "method": "Manual delta-delta",
        "estimate": messy_manual_summary.query('quantity == "delta-delta: drug genotype gap - placebo genotype gap"')["value"].iloc[0],
        "ci_low": np.nan,
        "ci_high": np.nan,
        "p_value": np.nan,
        "uncertainty_source": "Direct arithmetic on observed means",
    },
    {
        "method": "DABEST delta-delta",
        "estimate": messy_row["difference"] if "difference" in messy_row.index else np.nan,
        "ci_low": messy_row["bca_low"] if "bca_low" in messy_row.index else np.nan,
        "ci_high": messy_row["bca_high"] if "bca_high" in messy_row.index else np.nan,
        "p_value": messy_row["pvalue_permutation"] if "pvalue_permutation" in messy_row.index else np.nan,
        "uncertainty_source": "Bootstrap CI from resampled effect-size distribution",
    },
    {
        "method": "OLS interaction beta",
        "estimate": messy_model.params[messy_interaction_term],
        "ci_low": messy_reg_ci_low,
        "ci_high": messy_reg_ci_high,
        "p_value": messy_model.pvalues[messy_interaction_term],
        "uncertainty_source": "Model-based SE and t distribution",
    },
    {
        "method": "ANOVA interaction, Type II",
        "estimate": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "p_value": messy_anova_type2.loc[find_interaction_term(messy_anova_type2.index), "PR(>F)"],
        "uncertainty_source": "Model-based F test",
    },
])

print("Messy data comparison:")
display(messy_comparison)

print("ANOVA Type I:")
display(messy_anova_type1)

print("ANOVA Type II:")
display(messy_anova_type2)

print("ANOVA Type III:")
display(messy_anova_type3)


## 11. When the numerical identity breaks

The raw delta-delta and OLS interaction beta match cleanly when the model is a saturated 2x2 OLS model and the coefficient coding matches the desired contrast.

They stop meaning exactly the same thing when you change the model.

For example, if you add a covariate, the interaction beta is now an **adjusted** interaction estimate. It is no longer simply the raw difference between observed cell means.


In [ ]:
# Add a made-up covariate that is correlated with genotype/treatment enough to change the model.
rng = np.random.default_rng(42)
df_covariate = df_delta2_drug1.copy()
df_covariate["Body_mass"] = (
    1.0
    + 0.2 * (df_covariate["Genotype"].astype(str) == "M").astype(float)
    - 0.1 * (df_covariate["Treatment"].astype(str) == "Drug").astype(float)
    + rng.normal(0, 0.15, len(df_covariate))
)

raw_model = ols(formula, data=df_covariate).fit()

adjusted_formula = (
    'Survival_years ~ Body_mass + '
    'C(Genotype, levels=["W", "M"]) * C(Treatment, levels=["Placebo", "Drug"])'
)

adjusted_model = ols(adjusted_formula, data=df_covariate).fit()

raw_interaction_term = find_interaction_term(raw_model.params.index)
adjusted_interaction_term = find_interaction_term(adjusted_model.params.index)

breaks_table = pd.DataFrame([
    {
        "analysis": "Manual raw delta-delta",
        "estimate": manual_dd,
        "what_it_means": "Unadjusted difference between observed cell means",
    },
    {
        "analysis": "Saturated OLS interaction",
        "estimate": raw_model.params[raw_interaction_term],
        "what_it_means": "Same raw 2x2 contrast, if coding matches",
    },
    {
        "analysis": "OLS interaction adjusted for Body_mass",
        "estimate": adjusted_model.params[adjusted_interaction_term],
        "what_it_means": "Adjusted interaction; no longer simple raw cell-mean arithmetic",
    },
])

display(breaks_table)


## 12. Interpretation

The clean lesson:

1. The **manual delta-delta** is an arithmetic contrast.
2. In a matching saturated 2x2 OLS model, the **interaction beta** estimates the same contrast.
3. A two-way **ANOVA interaction p value** tests the corresponding no-interaction null.
4. The **DABEST bootstrap CI** is different in spirit from the model-based regression CI because it is built by resampling the observed data and recomputing the effect size.
5. The relationship becomes less simple when the model changes: unbalanced designs, Type I/II/III ANOVA choices, contrast coding, covariates, clustering, mixed effects, GLMs, or transformed outcomes.

The blog version of this is:

> The mean is the estimate; the interval is the story of how much we trust it.


## References

- DABEST Python documentation: <https://acclab.github.io/DABEST-python/>
- DABEST bootstrap confidence intervals blog post: <https://acclab.github.io/DABEST-python/blog/posts/bootstraps/bootstraps.html>
- DABEST loading data API, including `delta2`, `experiment`, `resamples`, and `random_seed`: <https://acclab.github.io/DABEST-python/API/load.html>
- Efron, B. (1979). *Bootstrap Methods: Another Look at the Jackknife*. The Annals of Statistics.
- Gardner, M. J. & Altman, D. G. (1986). *Confidence intervals rather than P values: estimation rather than hypothesis testing*. BMJ.
- Cumming, G. (2014). *The new statistics: why and how*. Psychological Science.
